# Qwen3.8-35B-A3B — full Colab

This notebook **clones the repo**, installs it, then distills **Qwen3.8-27B** onto the **Qwen3.6-35B-A3B** MoE runtime.

1. `git clone` + `pip install -e ".[colab]"`
2. Overlay Qwen3.8 configs
3. NF4 teacher logits from `Qwen/Qwen3.8-27B`
4. LoRA-distill `Qwen/Qwen3.6-35B-A3B`
5. Export HF serve config and save to Drive
6. Optional: merge LoRA and write Unsloth-style `UD-Q4_K_XL` / `UD-Q3_K_XL` GGUFs

**Runtime → A100 (40GB+).** Hugging Face login is interactive if you have not added a Colab secret named `HF_TOKEN`. Add `GITHUB_TOKEN` only if the repo is private.

Open from GitHub: [Open in Colab](https://colab.research.google.com/github/birdup000/qwen3-8-35b-a3b/blob/main/notebooks/Qwen3.8-35B-A3B_Colab.ipynb)

## 1. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), "Runtime → Change runtime type → A100 GPU."
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024**3
print(props.name, f"{vram:.1f} GB", "torch", torch.__version__)
if vram < 22:
    raise RuntimeError(f"Need ~22GB+ VRAM for 4-bit two-phase KD; this GPU has {vram:.1f} GB.")

## 2. Clone the repo and install

This cell is self-contained. You do **not** upload a zip. It clones `birdup000/qwen3-8-35b-a3b` and runs setup.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/birdup000/qwen3-8-35b-a3b.git"
REPO_BRANCH = "main"
DEST = Path("/content/Qwen3.8-35B-A3B")

try:
    from google.colab import userdata
    os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
except Exception:
    pass

token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN") or ""
clone_url = REPO_URL
if token and REPO_URL.startswith("https://") and "@" not in REPO_URL[8:]:
    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1)

def sh(cmd, cwd=None):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=cwd)

if (DEST / ".git").is_dir():
    sh(["git", "fetch", "--depth", "1", "origin", REPO_BRANCH], cwd=DEST)
    sh(["git", "checkout", REPO_BRANCH], cwd=DEST)
    sh(["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=DEST)
elif not (DEST / "qwen3_8_moe" / "configuration.py").is_file():
    DEST.parent.mkdir(parents=True, exist_ok=True)
    sh(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, str(DEST)])
else:
    print("Repo files already present")

os.chdir(DEST)
if str(DEST) not in sys.path:
    sys.path.insert(0, str(DEST))

sh([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
sh([sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"], cwd=DEST)

from qwen3_8_moe import parameter_report, qwen38_35b_a3b_config
text = qwen38_35b_a3b_config().text_config
assert (text.hidden_size, text.num_hidden_layers, text.num_experts) == (2048, 40, 256)
print("Cloned", DEST)
print(f"Graph {text.num_hidden_layers}L / {text.num_experts}E  active~{parameter_report()['active']/1e9:.2f}B")
print(sorted(p.name for p in DEST.iterdir() if not p.name.startswith(".")))

## 3. Hugging Face login + Drive

If you have no Colab secret, this cell opens a Hugging Face login widget. Create a token at https://huggingface.co/settings/tokens (read access is enough). Accept the licenses on [Qwen3.6-35B-A3B](https://huggingface.co/Qwen/Qwen3.6-35B-A3B) and [Qwen3.8-27B](https://huggingface.co/Qwen/Qwen3.8-27B).

To skip the widget next time: Colab left sidebar → 🔑 **Secrets** → add `HF_TOKEN` → enable Notebook access.

In [ ]:
import os
from pathlib import Path
from google.colab import drive
from huggingface_hub import login, notebook_login

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

if token:
    os.environ["HF_TOKEN"] = token
    login(token=token)
    print("Logged in with HF_TOKEN")
else:
    print("No Colab secret named HF_TOKEN.")
    print("Paste a token from https://huggingface.co/settings/tokens in the widget.")
    notebook_login()

drive.mount("/content/drive")
DRIVE_OUT = Path("/content/drive/MyDrive/Qwen3.8-35B-A3B")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("Drive ready at", DRIVE_OUT)

## 4. Confirm the 35B-A3B graph

In [ ]:
from scripts.colab_pipeline import detect_runtime, ensure_repo
from qwen3_8_moe import parameter_report, qwen38_35b_a3b_config

ensure_repo()
print(detect_runtime())
report = parameter_report()
print(f"Total {report['total'] / 1e9:.3f}B   active {report['active'] / 1e9:.3f}B")
print("Serve as", qwen38_35b_a3b_config().to_hf_dict()["architectures"][0])

## 5. Full distill

Built-in coding/reasoning prompts, or set `DATA_PATH` to a jsonl of `{"text": "..."}` lines.

In [ ]:
from pathlib import Path
from scripts.colab_pipeline import run_full_pipeline

TEACHER_ID = "Qwen/Qwen3.8-27B"
STUDENT_ID = "Qwen/Qwen3.6-35B-A3B"
WORK = Path("/content/qwen38_work")
DATA_PATH = None  # or Path("/content/drive/MyDrive/qwen38_distill.jsonl")

paths = run_full_pipeline(
    teacher_id=TEACHER_ID,
    student_id=STUDENT_ID,
    work_dir=WORK,
    data_path=DATA_PATH,
    fourbit=True,
    seq_len=512,
    max_samples=64,
    steps=100,
    lr=1e-4,
    temperature=2.0,
)
print({key: str(value) for key, value in paths.items()})

## 6. Copy tokenizer + save to Drive

In [ ]:
import shutil
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID, trust_remote_code=True)
for dest in (paths["hf"], paths["lora"], paths["student"]):
    tokenizer.save_pretrained(dest)

for name in ("hf", "lora", "student"):
    shutil.copytree(paths[name], DRIVE_OUT / name, dirs_exist_ok=True)
print("Saved to", DRIVE_OUT)
print(sorted(p.name for p in DRIVE_OUT.iterdir()))

## 7. Serve like Qwen3.6

HF / vLLM after a full merge (section 8 also merges for GGUF):

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.6-35B-A3B", device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, "/content/drive/MyDrive/Qwen3.8-35B-A3B/lora").merge_and_unload()
model.save_pretrained("/path/to/Qwen3.8-35B-A3B-merged")
```

```bash
vllm serve /path/to/Qwen3.8-35B-A3B-merged
```

```python
extra_body = {
    "chat_template_kwargs": {
        "enable_thinking": True,
        "preserve_thinking": True,
        "reasoning_effort": "xhigh",
    }
}
```

llama.cpp / Ollama after section 8:

```bash
./llama-cli -m Qwen3.8-35B-A3B-UD-Q4_K_XL.gguf -ngl 99 -ot ".ffn_.*_exps.=CPU"
```

## 8. GGUF — Unsloth XL (`UD-Q4_K_XL` / `UD-Q3_K_XL`)

Run this **after distill works**. It merges the LoRA into `Qwen/Qwen3.6-35B-A3B`, converts with llama.cpp, then quantizes with the public Unsloth/Bartowski XL recipe (Q8 embeddings + output, Q8 `ssm_out`, higher-bit `ffn_down`).

Default file: `Qwen3.8-35B-A3B-UD-Q4_K_XL.gguf` (~20GB). Set `QUANTS = ["Q3_K_XL"]` or both.

**Disk:** High-RAM runtime. Peak is ~70GB merged HF + ~67GB BF16 GGUF. The cell deletes the 27B teacher cache first. BF16 GGUF is removed after the XL file is written.

In [ ]:
import os
from pathlib import Path

# After a kernel restart, recover paths from Drive / work dir.
WORK = Path(globals().get("WORK", "/content/qwen38_work"))
DRIVE_OUT = Path(globals().get("DRIVE_OUT", "/content/drive/MyDrive/Qwen3.8-35B-A3B"))
STUDENT_ID = globals().get("STUDENT_ID", "Qwen/Qwen3.6-35B-A3B")
_paths = globals().get("paths") or {}
LORA_DIR = Path(_paths.get("lora", DRIVE_OUT / "lora"))
if not (LORA_DIR / "adapter_config.json").is_file():
    raise FileNotFoundError(f"No LoRA adapter at {LORA_DIR}. Run the distill cell first.")

# Q4_K_XL ~20GB (recommended). Q3_K_XL ~16GB. Use both if disk allows.
QUANTS = ["Q4_K_XL"]

# cmake + a compiler for llama-quantize / llama-imatrix
os.system("apt-get update -qq && apt-get install -y -qq build-essential cmake git >/dev/null")

from scripts.colab_pipeline import export_unsloth_xl_gguf

gguf_paths = export_unsloth_xl_gguf(
    work_dir=WORK / "gguf",
    lora_dir=LORA_DIR,
    out_dir=DRIVE_OUT,
    quants=QUANTS,
    base_id=STUDENT_ID,
    free_teacher=True,
    imatrix=True,
    keep_bf16=False,
)
print({key: str(value) for key, value in gguf_paths.items()})
print("GGUF files:", sorted(p.name for p in DRIVE_OUT.glob("*.gguf")))